In [1]:
!pip install -q moviepy
!pip install -q llama-index langchain
!pip install -q llama-index-embeddings-huggingface
!pip install streamlit
!npm install localtunnel
!pip install --quiet pyngrok
!pip install vidmaker

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf 23.8.0 requires cubinlinker, which is not installed.
cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
cudf 23.8.0 requires ptxcompiler, which is not installed.
cuml 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
dask-cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
keras-cv 0.8.2 requires keras-core, which is not installed.
keras-nlp 0.9.3 requires keras-core, which is not installed.
tensorflow-decision-forests 1.8.1 requires wurlitzer, which is not installed.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.8 which is incompatible.
apache-beam 2.46.0 requires numpy<1.25.0,>=1.14.3, but you have numpy 1.26.4 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 15.0.2 which is incompatible.
cudf

In [2]:
from tqdm import tqdm
import os
from PIL import Image
import cv2
import moviepy.editor as mp
import zipfile
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from langchain.text_splitter import RecursiveCharacterTextSplitter
from llama_index.core.node_parser import LangchainNodeParser

In [3]:
%%writefile app.py

import os
import streamlit as st
import subprocess
import cv2
import numpy as np
import streamlit as st
import base64
import moviepy.editor as mp
from tqdm import tqdm
import os
import numpy as np
from PIL import Image
import traceback
import cv2
import moviepy.editor as mp
import zipfile
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from langchain.text_splitter import RecursiveCharacterTextSplitter
from llama_index.core.node_parser import LangchainNodeParser
from transformers import pipeline
from io import BytesIO
import base64
import os
from datetime import datetime, timedelta
import subprocess
import vidmaker
import time
import subprocess

def convert_video(input_path, output_path):
    """
    Convert a video file to a format compatible with Streamlit using ffmpeg.
    
    Parameters:
    - input_path: str, path to the input video file.
    - output_path: str, path where the output video file will be saved.
    
    Returns:
    - bool, True if the conversion was successful, False otherwise.
    """
    output_dir = os.path.dirname(output_path)
    os.makedirs(output_dir, exist_ok=True)
    
    command = [
        'ffmpeg',
        '-y', # to overwrite the existing one
        '-i', input_path,
        '-vcodec', 'libx264',
        output_path
    ]
    
    try:
        result = subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
#         st.success("FFmpeg output:")
#         st.success(result.stdout.decode())
#         st.success("FFmpeg error (if any):")
#         st.success(result.stderr.decode())
        return True
    except Exception as e:
        print(f"Error converting video: {e}")
#         st.success(traceback.format_exc())
#         st.success(e.stderr.decode()) 
        return False

def create_video(frames, output_path, fps):
    height, width, _ = frames[0].shape

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    for frame in frames:
        out.write(frame)

    out.release()

    with open("/kaggle/working/OutputVideoFolder/outputvideo.mp4", "rb") as file:
        btn = st.download_button(
                label="Download Video",
                data=file,
                file_name="outputvideo.mp4",
                mime="video/mp4"
              )
    
    input_video_path = '/kaggle/working/OutputVideoFolder/outputvideo.mp4'
    output_video_path = 'kaggle/working/OutputVideoFolder/output_video_that_streamlit_can_play.mp4'

    
    
    # Convert the video using ffmpeg
    if convert_video(input_video_path, output_video_path):
        st.title("Check out your Summarized Video below... ")
        st.success("Summarized Video")
        # Display the video in Streamlit
        st.video(output_video_path)
    else:
        st.error("Failed to convert video.")
        
    
    
    
def frame_generator(video_path, start_frame, end_frame):
    video = cv2.VideoCapture(video_path)
    for idx in range(start_frame, end_frame):
        video.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = video.read()
        if ret:
            yield frame
    video.release()
    
def extract_frames(frame_number, video_path):
    e_frames = [frame for frame in frame_generator(video_path, frame_number - 120, frame_number + 120)]
   # st.success("Frames extracted:" + str(len(e_frames)))
    return e_frames

def generate_secure_link(file_path, expiration_time_hours=1):
    with open(file_path, "rb") as file:
        file_bytes = file.read()
    encoded_file = base64.b64encode(file_bytes).decode('utf-8')
    token = base64.urlsafe_b64encode(os.urandom(16)).decode('utf-8')
    expiration = datetime.utcnow() + timedelta(hours=expiration_time_hours)
    link = f"data:video/mp4;base64,{encoded_file}"
    return link, token, expiration

def download_video():
    video_path = "/kaggle/working/OutputVideoFolder/outputvideo.mp4"
    secure_link, token, expiration = generate_secure_link(video_path)


    if st.button("Generate Secure Download Link"):
        secure_link, token, expiration = generate_secure_link(video_path)
        download_url = f"/download?token={token}"
        st.markdown(f"### [Download Video]({download_url})")
       # st.success(f"Secure link generated! Click the link to download the video.")
        st.write(f"Expires at: {expiration}")

def main():
    video_path = ''
    st.title("Query Based Video Summarization - FYP-II")
    
    # loading trained model
    image_captioner = pipeline("image-to-text", model="/kaggle/input/fyp-dataset-ego4d/image-captioning-output", device = 'cuda')

    # loading embeddings model
    embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-small-en-v1.5",
        device='cuda'
    )
    # Text input for user to enter a string
    user_string = st.text_input("Enter a string:")
    userinputquery = user_string

    # File uploader for user to upload a video
    uploaded_file = st.file_uploader("Upload a video (MP4 format)", type=["mp4"])

    # Create 'kaggle/working' directory if it doesn't exist
    output_folder = '/kaggle/working/inputVideos'
    os.makedirs(output_folder, exist_ok=True)
    
    # Check if both video and string are uploaded
    if uploaded_file is not None and user_string:
        # Save video file to 'kaggle/working' directory
        with open(os.path.join(output_folder, "uploaded_video.mp4"), "wb") as f:
            f.write(uploaded_file.read())
            
            ##Total Frames
        video_path = '/kaggle/working/inputVideos/uploaded_video.mp4'
        clip = mp.VideoFileClip(video_path)
        frame_rate = clip.fps
        duration = clip.duration
        total_frames = int(frame_rate * duration)
        clip.close()
        
        #Extracr Frames
        video_id = video_path.split("/")[-1].split(".")[0]
#        st.success("Video id = " + str(video_id))
        clip = mp.VideoFileClip(video_path)
        frame_rate = clip.fps
        total_frames = int(clip.duration * frame_rate)
        
       # st.success(total_frames)

        for i in tqdm(range(0, total_frames, 15)):
            frame_number = i
            frame = clip.get_frame(i / frame_rate)
            frame_path = f"{output_folder}{video_id}_{frame_number}.jpg"
            img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            img.save(frame_path)

        clip.close()

        # Save user string to a text file in 'kaggle/working' directory
        with open(os.path.join(output_folder, "user_string.txt"), "w") as f:
            f.write(user_string)

            st.success("Contents uploaded successfully, Processing it ...")
        
        folder_path = "/kaggle/working/"
        os.makedirs(folder_path, exist_ok=True)
        files = os.listdir(folder_path)
        image_files = [file for file in files if file.endswith((".jpg", ".jpeg", ".png", ".gif"))]
        test_image_paths = [os.path.join(folder_path, file) for file in image_files]  
        # st.success("Test image path = " + str(test_image_paths))
    
        pred = {}
        prev_caption = None
        for i in tqdm(range(len(test_image_paths))):
            caption = image_captioner(test_image_paths[i])
            if caption != prev_caption:
                prev_caption = caption
                frame_number_str = test_image_paths[i].split("_")[-1]
                frame_number_str = frame_number_str.split(".")[0]
                pred[int(frame_number_str)] = caption

        sorted_dict = {k: pred[k] for k in sorted(pred)}
        # st.success("sorted_dict = " + str(sorted_dict))
    
        documents = []
        for key, values in sorted_dict.items():
            for value in values:
                frame_number = key
                text = value['generated_text']
                document = Document(
                    text=text,
                    metadata={'frame': frame_number},
                    metadata_template="{key}=>{value}",
                )
                documents.append(document)
      #  st.success("documents = " + str(documents))
        
        Settings.embed_model = embed_model
        parser = LangchainNodeParser(RecursiveCharacterTextSplitter())
        nodes = parser.get_nodes_from_documents(documents)
    
        retriever = VectorStoreIndex(
            nodes
        ).as_retriever(similarity_top_k = 20)
    
        response = retriever.retrieve(userinputquery)

        r_frames = []
        for r in response:
           # st.warning(r.text, r.metadata)
            r_frames.append(r.metadata['frame'])
    
        r_frames = sorted(r_frames)
        i = 0
        for i in tqdm(range(0,len(r_frames)-1)):
            forward_diff = r_frames[i] + 120
            backward_diff = r_frames[i + 1] - 120
            if forward_diff > backward_diff:
                difference = abs((r_frames[i+1]) - r_frames[i] + 120)
                r_frames[i+1] += difference + 120
            i += 1
        
        r_frames = list(dict.fromkeys(r_frames))
        top_10_frames = r_frames[0:10]
    
        e_frames = list()
        for i in tqdm(range(len(top_10_frames))):
            frames = extract_frames(top_10_frames[i], video_path)
            e_frames += frames
    
      #  st.success(len(e_frames))
    
        output_folder_of_Video = '/kaggle/working/OutputVideoFolder'
        os.makedirs(output_folder_of_Video, exist_ok=True)
        
        output_path = '/kaggle/working/OutputVideoFolder/outputvideo.mp4'
        fps = 30
        create_video(e_frames, output_path, fps)
        
        # st.success(type(uploaded_file))
        st.success("Original Video")
        st.video(uploaded_file)
        
    elif uploaded_file is not None:
        st.warning("Please enter a string.")

    elif user_string:
        st.warning("Please upload a video.")
    
            
if __name__ == "__main__":
    main()


Writing app.py


In [4]:
!wget -q -O - ipv4.icanhazip.com

35.227.155.2


In [5]:
!streamlit run app.py --server.maxUploadSize=5000 &>/logs.txt & npx localtunnel --port 8501

your url is: https://loud-otters-change.loca.lt
/kaggle/working/node_modules/localtunnel/bin/lt.js:81
    throw err;
    ^

Error: connection refused: localtunnel.me:39453 (check your firewall settings)
    at Socket.<anonymous> (/kaggle/working/node_modules/localtunnel/lib/TunnelCluster.js:52:11)
    at Socket.emit (node:events:514:28)
    at emitErrorNT (node:internal/streams/destroy:151:8)
    at emitErrorCloseNT (node:internal/streams/destroy:116:3)
    at process.processTicksAndRejections (node:internal/process/task_queues:82:21)

Node.js v20.9.0
